# Baseline Classifier

Out-of-sample, walk-forward logistic regression predicting the **quantile / binary `label`**
built on the Feature Engineering page. All logic lives in `irp.models.classifier`.

**The dataset comes from `/features`**: build features + a **Quantile** (or Up/Down) label,
then **Export parquet**. This notebook loads that export — it does not rebuild the data.

In [1]:
from irp.models import classifier as clf
from sklearn.linear_model import LogisticRegression

# ── parameters ──
EXPORT_PATH = 'features_20260601_034025.parquet'        # None = most recent; or: clf.list_exports().iloc[1]['file']
FEATURE_COLS = None       # None = infer (numeric cols except Date/Ticker/fwd_ret/label)
MODEL = LogisticRegression(max_iter=1_000)
MIN_TRAIN_DATES = 12
TPL = clf.nb_template()

clf.list_exports()        # row 0 = newest — set EXPORT_PATH to pick a different one

,file,modified,mb
0,fe_dataset_20260607_100915_test.parquet,2026-06-07 10:09:20.175989,95.18
1,fe_dataset_20260607_100915_valid.parquet,2026-06-07 10:09:19.375986,93.14
2,fe_dataset_20260607_100915_train.parquet,2026-06-07 10:09:18.404982,390.20
3,fe_dataset_20260607_031306_test.parquet,2026-06-07 03:13:27.616632,577.08
4,fe_dataset_20260607_031306_valid.parquet,2026-06-07 03:13:23.489620,487.64
5,fe_dataset_20260607_031306_train.parquet,2026-06-07 03:13:18.938607,1598.46
6,features_20260606_194043.parquet,2026-06-06 19:41:00.137273,1653.04
7,features_20260606_135615_test.parquet,2026-06-06 13:56:34.423560,577.26
8,features_20260606_135615_valid.parquet,2026-06-06 13:56:30.231589,486.98
9,features_20260606_135615_train.parquet,2026-06-06 13:56:26.337615,1595.80


## 1 · Load dataset (must carry a quantile/binary `label`)

In [2]:
EXPORT_PATH = 'fe_dataset_20260607_100915_test.parquet'        # None = most recent; or: clf.list_exports().iloc[1]['file']


df, features = clf.load_export(EXPORT_PATH, feature_cols=FEATURE_COLS, target='label')
print(len(features), 'features:', features)
df['label'].value_counts().sort_index()

loaded fe_dataset_20260607_100915_test.parquet  706,771 rows × 43 cols
39 features: ['gross_margin', 'op_margin', 'net_margin', 'roe', 'roa', 'roic', 'fcf_margin', 'asset_turnover', 'cfo_ni_ratio', 'accruals', 'rsi_14', 'macd_hist', 'macd_norm', 'bb_pct', 'ma7_ma28', 'ma14_ma56', 'revenue', 'net_income', 'total_assets', 'total_equity', 'op_cashflow', 'rand', 'rev_growth_1y', 'earn_growth_1y', 'debt_equity', 'net_debt_ebitda', 'interest_coverage', 'piotroski_fscore', 'close', 'close_lag1', 'close_lag2', 'close_lag3', 'close_lag4', 'close_lag5', 'close_lag6', 'close_lag7', 'close_lag8', 'close_lag9', 'close_lag10']


label
0.0    203322
1.0    203269
2.0    203304
Name: count, dtype: int64

In [3]:
df

,Date,Ticker,gross_margin,op_margin,net_margin,roe,roa,roic,fcf_margin,asset_turnover,...,close_lag3,close_lag4,close_lag5,close_lag6,close_lag7,close_lag8,close_lag9,close_lag10,fwd_ret,label
0,2025-10-03,A,0.319312,0.604967,0.658412,0.461345,0.532579,0.384436,0.688475,-0.06778,...,1.644527,1.602472,1.598910,1.589010,1.604424,1.617641,1.616423,1.618948,-0.033925,0.0
1,2025-10-06,A,0.319312,0.604967,0.658412,0.461345,0.532579,0.384436,0.688475,-0.06778,...,1.731147,1.643999,1.601845,1.598493,1.588668,1.604219,1.617526,1.616314,-0.033714,0.0
2,2025-10-07,A,0.319312,0.604967,0.658412,0.461345,0.532579,0.384436,0.688475,-0.06778,...,1.732367,1.732672,1.644764,1.602598,1.598871,1.589547,1.604611,1.617676,-0.011940,0.0
3,2025-10-08,A,0.319312,0.604967,0.658412,0.461345,0.532579,0.384436,0.688475,-0.06778,...,1.755205,1.731653,1.731696,1.644445,1.602376,1.598811,1.589177,1.604495,-0.028048,0.0
4,2025-10-09,A,0.319312,0.604967,0.658412,0.461345,0.532579,0.384436,0.688475,-0.06778,...,1.752266,1.754700,1.731592,1.731239,1.644055,1.601944,1.598593,1.588873,-0.023065,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
706766,2025-12-25,^STI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.986317,5.984022,5.988399,5.995507,5.985800,5.987861,5.982247,5.956853,NaN,NaN
706767,2025-12-26,^STI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.996894,5.987072,5.985552,5.990022,5.996922,5.988505,5.989574,5.983848,NaN,NaN
706768,2025-12-29,^STI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.990440,5.995140,5.986139,5.983900,5.987961,5.995261,5.985693,5.987524,NaN,NaN
706769,2025-12-30,^STI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.990948,5.990948,5.995299,5.986366,5.983976,5.988352,5.995303,5.985724,NaN,NaN


## 2 · Walk-forward classification
Expanding window: fit on the past, predict the current cross-section's class (no look-ahead).

In [4]:
res = clf.walk_forward_classifier(df, features, target='label', model=MODEL, min_train_dates=MIN_TRAIN_DATES)
_ = clf.summary(res)

accuracy                      0.523078
baseline_acc (majority)       0.426668
n_classes                     3.000000
n_dates                      51.000000
n_preds                  102865.000000
score_IC                      0.283689
score_ICIR                    8.692128


## 3 · Visualize
Confusion + accuracy. Quintiles by predicted score need `fwd_ret` in the export (Quantile mode keeps it).

In [5]:
clf.plot_confusion(res, TPL)

In [6]:
clf.plot_accuracy(res, TPL)

In [7]:
clf.plot_quintiles(res, TPL)